# Phase 2 & 3 : Apprentissage Profond (Keras Tokenizer + Bidirectional LSTM)

Ce notebook présente la mise en œuvre d'une architecture de Deep Learning (réseau de neurones récurrents de type LSTM) pour la classification des sentiments sur la plateforme **BrandPulse AI**.

In [6]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Ajout du dossier parent au path pour importer preprocessing
import sys
sys.path.append('..')
from preprocessing import clean_tweet

print("TensorFlow Version:", tf.__version__)
print("Importations terminées avec succès.")

ModuleNotFoundError: No module named 'tensorflow'

## 1. Chargement et Nettoyage des Données

Nous rechargeons le dataset `Tweets.csv` précédemment téléchargé et effectuons le nettoyage.

In [ ]:
data_path = '../data/Tweets.csv'
if not os.path.exists(data_path):
    raise FileNotFoundError("Veuillez d'abord exécuter le notebook 01 pour télécharger le dataset.")

df = pd.read_csv(data_path)
print("Nettoyage des tweets...")
df['clean_text'] = df['text'].apply(clean_tweet)
df = df[df['clean_text'].str.strip() != '']
print(f"Taille après nettoyage : {df.shape[0]} tweets.")

Nettoyage des tweets...


NameError: name 'clean_tweet' is not defined

## 2. Encodage des Labels & Tokenisation

Les labels textuels (`negative`, `neutral`, `positive`) doivent être convertis en catégories numériques encodées en One-Hot pour correspondre à la sortie Softmax de notre réseau.

In [ ]:
# Mapping des étiquettes
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['airline_sentiment'].map(label_map)

# Tokenisation du texte
max_features = 10000  # Nombre maximum de mots uniques à conserver
tokenizer = Tokenizer(num_words=max_features, split=' ', oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'].values)

# Sauvegarde du Tokenizer pour utilisation ultérieure dans le Dashboard
os.makedirs('../models', exist_ok=True)
with open('../models/tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Tokenizer entraîné et sauvegardé avec succès.")

NameError: name 'Tokenizer' is not defined

## 3. Padding des Séquences & Division Train/Test

In [ ]:
# Conversion du texte en séquences numériques
X_seq = tokenizer.texts_to_sequences(df['clean_text'].values)

# Remplissage (padding) des séquences pour avoir une longueur fixe
max_len = 40  # Longueur maximale d'un tweet nettoyé
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')

# Cibles One-Hot encoded
y_cat = to_categorical(df['label'].values, num_classes=3)

# Séparation stratifiée en train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_pad, y_cat, test_size=0.2, random_state=42, stratify=df['label'].values)
print(f"Forme des données d'entraînement : {X_train.shape}")
print(f"Forme des données de test : {X_test.shape}")

## 4. Construction de l'Architecture LSTM

Notre architecture comprend :
1. Une couche **Embedding** pour apprendre la représentation vectorielle dense des mots.
2. Une couche **SpatialDropout1D** pour régulariser l'Embedding.
3. Une couche **Bidirectional LSTM** pour extraire le contexte dans les deux sens de lecture.
4. Une couche **Dense** intermédiaire avec Dropout.
5. Une couche **Dense finale** avec Softmax pour la classification finale en 3 classes.

In [ ]:
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=max_features, output_dim=embedding_dim, input_length=max_len),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 5. Entraînement du Modèle

Nous configurons un callback d'**Early Stopping** pour stopper l'entraînement si la perte de validation cesse de s'améliorer, ce qui évite le surapprentissage.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

epochs = 8
batch_size = 64

print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

## 6. Évaluation du Modèle & Courbes de Perte

In [ ]:
# Tracé des courbes d'exactitude et de perte
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Perte
ax1.plot(history.history['loss'], label='Train Loss')
ax1.plot(history.history['val_loss'], label='Val Loss')
ax1.set_title('Évolution de la Perte (Loss)')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

# Exactitude
ax2.plot(history.history['accuracy'], label='Train Accuracy')
ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
ax2.set_title('Évolution de l\'Exactitude (Accuracy)')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.savefig('../data/learning_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Prédiction sur l'ensemble de test
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Inversion du mapping pour le rapport de classification
target_names = ['negative', 'neutral', 'positive']

print("\n=== RAPPORT : LSTM DEEP LEARNING ===")
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

In [ ]:
# Matrice de confusion
plt.figure(figsize=(7, 6))
cm = confusion_matrix(y_true_classes, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap='Purples', ax=plt.gca())
plt.title('Matrice de Confusion : Modèle LSTM')
plt.savefig('../data/confusion_matrix_lstm.png', bbox_inches='tight')
plt.show()

## 7. Sauvegarde du Modèle LSTM

In [ ]:
# Sauvegarde du modèle au format Keras moderne
model.save('../models/lstm_model.keras')
print("Modèle Keras LSTM sauvegardé sous '../models/lstm_model.keras'.")